# Uber "Find My Ride" - Secondary Review Analysis (pilot sample)
**Window requested:** 17 Apr - 17 Oct 2026. **Dates actually obtainable:** 12 Sep - 14 Oct 2026 (see Limitations).
**Source:** Apple App Store, India storefront, via Apple's public customer-reviews RSS feed - the only source whose review text and original dates could be fetched. Google Play and Reddit were blocked or unrendered.
**Sample:** 33 reviews screened as relevant in `Secondary_Review_Screening.csv`, matched by review_id to the feed; dates verified from the feed's `updated` field.
**Not representative.** App-store reviewers self-select toward dissatisfaction; this is a purposive sample for theme discovery, not a prevalence estimate for Uber users. Primary and secondary percentages use different denominators and are not statistically comparable.

In [ ]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
sec = pd.read_csv(ROOT/'data/clean/Secondary_Reviews_Verified.csv', encoding='utf-8-sig', dtype={'review_id': str})
pri = pd.read_csv(ROOT/'data/clean/Cleaned_Primary_Research.csv'); pri = pri[pri.include_in_analysis == True]
assert len(sec) == 33 and sec.review_id.is_unique and len(pri) == 24
assert sec.pub_date.between('2026-04-17', '2026-10-17').all()
print(len(sec), 'secondary reviews;', sec.pub_date.min(), 'to', sec.pub_date.max(), '| primary n =', len(pri))

## 1. Sample description

In [ ]:
print(sec.stakeholder.value_counts().to_string()); print()
print('Star ratings:'); print(sec.rating.value_counts().sort_index().to_string()); print()
print('Reviews per week:'); print(pd.to_datetime(sec.pub_date).dt.to_period('W').value_counts().sort_index().to_string()); print()
print('Geography (as stated):'); print(sec.geography.value_counts().to_string())

## 2. Descriptive theme and journey-stage frequency
Multi-coding allowed (a review can carry several codes), so percentages sum to more than 100%. Codes use the project's seed codebook (`PROMPT.md`), with FARE_TRUST, FALSE_ARRIVAL and SCHEDULED_TIMING added from the data.

In [ ]:
th = sec.theme_codes.str.split(';').explode().value_counts()
freq = pd.DataFrame({'reviews': th, 'pct_of_33': (th/len(sec)*100).round(0).astype(int)})
freq['mean_rating'] = [round(sec[sec.theme_codes.str.contains(t)].rating.mean(), 1) for t in freq.index]
display(freq)
stg = sec.journey_stages.str.split(';').explode().value_counts().sort_index()
display(pd.DataFrame({'reviews': stg, 'pct_of_33': (stg/len(sec)*100).round(0).astype(int)}))
display(pd.crosstab(sec.theme_codes.str.split(';').explode(), sec.loc[sec.theme_codes.str.split(';').explode().index, 'stakeholder'].values))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
f = freq.sort_values('reviews')
ax.barh(f.index.str.replace('_', ' ').str.title(), f.reviews, color='#2a6f97')
for y, (v, p) in enumerate(zip(f.reviews, f.pct_of_33)): ax.text(v+.3, y, f'{v} ({p}%)', va='center', fontsize=9)
ax.set_xlabel('Reviews mentioning theme (n = 33; multi-coded)'); ax.set_xlim(0, f.reviews.max()+6)
ax.set_title('App-store reviews: cancellation dominates', loc='left', fontsize=12)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
plt.tight_layout(); (ROOT/'outputs').mkdir(exist_ok=True)
plt.savefig(ROOT/'outputs/secondary_theme_frequency.png', dpi=130); plt.close()

In [ ]:
order = ['1 Driver assigned', '2 Pickup interpretation', '3 Driver approaches', '4 Coordination', '5 Meeting', '6 Outcome']
s = stg.reindex(order).fillna(0)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([o.split(' ', 1)[1] for o in order], s.values, color='#2a6f97')
for x, v in enumerate(s.values): ax.text(x, v+.4, int(v), ha='center', fontsize=9)
ax.set_ylabel('Reviews touching stage (n = 33)'); ax.set_title('Where reviews place the friction (journey stage)', loc='left', fontsize=12)
plt.xticks(fontsize=8)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(ROOT/'outputs/secondary_journey_stage.png', dpi=130); plt.close()

## 3. Qualitative thematic analysis
Process: read all 33 reviews, apply seed codes, group into themes, pull verbatim evidence and counter-evidence. Observation (what the reviewer said) is kept apart from interpretation. Rider accounts are not used to infer driver motives.

In [ ]:
def q(rid, frag):
    r = sec[sec.review_id == rid].iloc[0]
    assert frag in r.review_text or frag in r.review_title, (rid, frag)
    return f'"{frag}" ({rid}, {r.pub_date})'
themes = pd.DataFrame([
 ['Driver cancels after accepting', 'CANCELLATION', q('14645034712', 'the drivers either cancel trips, ask you to cancel, or accept but never arrive'),
  q('14656670481', 'Good service') + ' (review title, 5 stars) - yet the text reports a cancellation-charge dispute', 'Acceptance is not treated as commitment, so riders cannot plan around a confirmed pickup.', 'High that riders report it; low on how often it happens'],
 ['Waiting with no information', 'WAITING; COMMUNICATION', q('14664449542', 'Your driver remained stuck just 2 minutes away'), q('14630334264', 'Driver corrected it') + ' - one case where the driver fixed a wrong route', 'The pain is silence during the approach, not only delay.', 'Medium'],
 ['Off-app fare demands at pickup', 'FARE_TRUST', q('14595321473', 'They ask after coming to the pick location and dont say this on the call or on the app msg.'), 'No contradicting review identified in this sample', 'Pickup becomes the moment price is renegotiated and overlaps with cancellation fees. Adjacent to, not inside, the find-my-ride problem.', 'Medium'],
 ['Pin shown differently to rider and driver', 'PIN_MISMATCH; DRIVER_LOCATION_DIFFICULTY', q('14555310975', 'The pickup and drop-off locations shown to us in the app were sometimes different from what was being shown to the drivers'), 'Fewer reviews than cancellation or waiting', 'Closest direct support for the Find My Ride premise, but thinly evidenced here.', 'Low-Medium'],
 ['Airport and gate confusion', 'GATE_CONFUSION; VEHICLE_IDENTIFICATION', q('14653532861', 'your pickup point is miles away from the gate, your directions are unclear'), q('14664679835', 'In Airport some help desk or location link to be added') + ' - a constructive 4-star suggestion', 'Airports concentrate walking, signage and touting problems; one review alleges a car swap at Cochin airport (unverified allegation).', 'Low (2 gate-coded reviews)'],
 ['Driver asks rider to come elsewhere', 'RIDER_WALKING', q('14661268058', 'They are not even willing to come to the given pickup location and keep asking customers to come to different places'), 'None in sample', 'Observation only; driver motive unknown.', 'Low'],
 ['Driver view: riders cancel or do not answer', 'CANCELLATION; COMMUNICATION (driver)', q('14640771178', 'because riders cancel'), 'Only one driver review in the sample', 'Cancellation is costly to both sides, but one review cannot establish a driver pattern.', 'Very low'],
], columns=['Theme', 'Codes', 'Supporting evidence (verbatim)', 'Contradictory / limiting evidence', 'Interpretation', 'Confidence'])
pd.set_option('display.max_colwidth', None); display(themes)

## 4. Primary vs secondary comparison (descriptive)
Primary = 24 valid Uber users (survey, multi-select questions). Secondary = 33 app-store reviews (multi-coded). Different instruments and denominators: **compare rank order and direction, not percentages.**

In [ ]:
P = lambda c: int(pri[c].fillna(0).astype(float).sum())
S = lambda t: int(sec.theme_codes.str.contains(t).sum())
cmp = pd.DataFrame([
 ['Gate / entrance confusion', P('Q6__confusion_about_which_entrance_or_gate_t'), S('GATE_CONFUSION')],
 ['Incorrect / inconvenient pickup pin', P('Q6__pickup_pin_showed_an_incorrect_or_inconv'), S('PIN_MISMATCH')],
 ['Long waiting', P('Q6__long_waiting_time'), S('WAITING')],
 ['Driver cannot find exact location', P('Q6__driver_could_not_find_my_exact_location'), S('DRIVER_LOCATION_DIFFICULTY')],
 ['Driver asked rider to walk elsewhere', P('Q6__driver_asked_me_to_walk_to_another_locat'), S('RIDER_WALKING')],
 ['Poor communication', P('Q7__did_not_communicate_clearly'), S('COMMUNICATION')],
 ['Driver cannot access pickup point', P('Q6__driver_could_not_access_the_pickup_point'), S('ACCESS_RESTRICTION')],
 ['Difficulty identifying vehicle', P('Q6__difficulty_identifying_the_driver_s_vehi'), S('VEHICLE_IDENTIFICATION')],
 ['Cancellation', int((pri.Q10 == 'Driver cancelled the booking').sum()), S('CANCELLATION')],
 ['Fare / price trust', np.nan, S('FARE_TRUST')],
], columns=['Theme', 'Primary n (of 24)', 'Secondary n (of 33)'])
cmp['Primary %'] = (cmp['Primary n (of 24)']/24*100).round(0); cmp['Secondary %'] = (cmp['Secondary n (of 33)']/33*100).round(0)
display(cmp)
print('Primary cancellation row = Q10 outcome "Driver cancelled the booking"; secondary = any review coded CANCELLATION (mostly driver-initiated).')
c = cmp.sort_values('Secondary %')
fig, ax = plt.subplots(figsize=(8.5, 4.8)); y = np.arange(len(c)); h = .38
ax.barh(y-h/2, c['Primary %'].fillna(0), h, label='Primary survey (n=24)', color='#2a6f97')
ax.barh(y+h/2, c['Secondary %'], h, label='App-store reviews (n=33)', color='#e07a1f')
ax.set_yticks(y); ax.set_yticklabels(c.Theme, fontsize=8)
ax.set_xlabel('% of respondents / reviews mentioning (not directly comparable)'); ax.legend(fontsize=8, frameon=False, loc='lower right')
ax.set_title('Different lenses, different emphasis: gate and pin vs cancellation and fare', loc='left', fontsize=11)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(ROOT/'outputs/primary_vs_secondary_themes.png', dpi=130); plt.close()

## 5. Triangulation matrix
Cell numbers are computed from the data above.

In [ ]:
pc = lambda n, d: f'{n} of {d} ({n/d:.0%})'
tri = pd.DataFrame([
 ['Long waiting', pc(P('Q6__long_waiting_time'), 24), pc(S('WAITING'), 33), 'PARTIAL CONVERGENCE', 'Waiting is a leading difficulty in both, but weighs about half as much in reviews; reviews add that silence during the wait is the pain.', 'Moderate', 'Measure wait length and what the rider sees meanwhile'],
 ['Communication failures', pc(P('Q7__did_not_communicate_clearly'), 24), pc(S('COMMUNICATION'), 33), 'CONVERGENCE', 'Unanswered calls and invalid numbers appear in both.', 'Moderate', 'Interview riders on call vs in-app chat'],
 ['Driver cancellation', 'Outcome "driver cancelled": ' + pc(int((pri.Q10 == 'Driver cancelled the booking').sum()), 24), pc(S('CANCELLATION'), 33), 'CONVERGENCE on existence; CONTRADICTION on size', 'Reviews over-represent cancellation (self-selection toward 1-star). Primary shows it is real but not the main outcome: 12 of 24 primary respondents reported meeting the driver (9 after a delay) and 4 reported driver cancellation.', 'Low on magnitude', 'Get platform cancellation data; do not read review share as incidence'],
 ['Gate / entrance confusion', pc(P('Q6__confusion_about_which_entrance_or_gate_t'), 24), pc(S('GATE_CONFUSION'), 33), 'CONTRADICTION in rank', 'Top primary difficulty, but only 2 reviews code it, both airport-related. Reviewers may not attribute failures to gates.', 'Low-Moderate', 'Code a larger review sample; test airports separately'],
 ['Incorrect pickup pin', pc(P('Q6__pickup_pin_showed_an_incorrect_or_inconv'), 24), pc(S('PIN_MISMATCH'), 33), 'CONTRADICTION in rank', 'Third-ranked primary difficulty but rarely raised in reviews (2 of 33); still the most direct support for the Find My Ride premise.', 'Low-Moderate', 'Add Play/Reddit reviews mentioning pin and location'],
 ['Fare / price trust at pickup', 'Not measured', pc(S('FARE_TRUST'), 33), 'GAP in primary', 'Raised unprompted in reviews and entangled with cancellation fees.', 'Moderate', 'Add fare-renegotiation items to next survey wave'],
 ['Vehicle identification / safety', pc(P('Q6__difficulty_identifying_the_driver_s_vehi'), 24), pc(S('VEHICLE_IDENTIFICATION'), 33), 'CONTRADICTION in salience', 'Rare in the survey, but reviews raise wrong-car and safety allegations.', 'Low', 'Probe safety severity in interviews'],
 ['Driver perspective', 'Not collected in survey', f"{int((sec.stakeholder == 'driver').sum())} of 33", 'GAP in both', 'No usable driver evidence; r/uberdrivers could not be accessed.', 'Very low', 'Driver interviews; supply r/uberdrivers posts'],
 ['Geography', 'See primary dataset', f"City stated in {int((~sec.geography.str.startswith('India (App')).sum())} of 33", 'GAP', 'Most reviews give no city, so city and airport comparisons are weak.', '-', 'Collect city-tagged sources'],
], columns=['Theme', 'Primary survey (n=24)', 'Secondary reviews (n=33)', 'Status', 'Reading', 'Confidence', 'Further research'])
display(tri)
tri.to_csv(ROOT/'outputs/Triangulation_Matrix.csv', index=False, encoding='utf-8-sig')

## 6. Limitations and access report
- **Source coverage:** Only Apple App Store (India) was retrievable. Google Play shows no review text without scripts; Reddit (r/uber, r/uberdrivers, r/bangalore) returned HTTP 403. No Reddit or Play reviews are included.
- **Date coverage:** Apple's feed returns only the latest 500 reviews (11 Sep - 14 Oct 2026). Nothing is available for 17 Apr - 10 Sep 2026.
- **Provenance:** Apple gives no per-review permalink; `source_url` is the feed and `app_listing_url` the public listing. `review_id` is Apple's own ID.
- **Screening:** 33 of the 57 'include' decisions in the screening file matched the feed; the other 24 included IDs were not used because their dates could not be verified. Theme codes come from the earlier screening; I re-read each text and changed one stakeholder label (14640771178 to driver).
- **Bias:** 1-2 star reviews dominate; reviewers cannot be verified as Uber riders; some reviews concern fare or support more than pickup. City is often not stated.
- **Not representative:** results describe these 33 reviews only.